In [1]:
import sys
sys.dont_write_bytecode = True
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import datetime as dt
from scipy.stats import pearsonr
import pickle
import torch
import torch.nn as nn
import os
import time
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import math
from math import sqrt
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import codes.mnn_Utils as mnn
from codes.make_dataset import DatasetHist
import random

from termcolor import colored

In [2]:
train_path = "../data/data_generated/train/train_ips_omni_df.csv"
val_path = "../data/data_generated/val/val_ips_omni_df.csv"
test_path = "../data/data_generated/test/test_ips_omni_df.csv"

In [3]:
full_path = "../data/data_generated/full_df.csv"

In [4]:
val_dataset = DatasetHist(val_path)

Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 71994 ends: 77399


In [32]:
len(val_dataset)

5406

In [67]:
val_load = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [68]:
model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=0.2, device=None)

In [71]:
for i, val in enumerate(val_load):
    print(val[1].shape, val[2].shape, val[1].to(torch.float32).dtype)
    model.eval()
    output = model(val[1].to(torch.float32))
    # print(output)
    loss = F.mse_loss(val[2], output)
    print(loss.item())
    if i > len(val_dataset)/100:
        break

torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.002494701088223222
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.002659302964279111
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.0053764669459863725
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.018937062894882666
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.031946096143575915
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.037891578355801085
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.03155205378294308
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.0240141916344229
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.014488449709021455
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.005001343805791681
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.0030746416131782013
torch.Size([16, 13, 32]) torch.Size([16, 1, 16]) torch.float32
0.002698214431

In [14]:
model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=0.2, device=None)
model.train()
# Inititialize optimizer ans scheduler
lr = lrs[0]
weight_decay = weight_decays[0]
batch_size = batch_sizes[0]
optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

data_set = DatasetHist(train_path)
data_load = DataLoader(data_set, batch_size=batch_size, shuffle=True)

for i, batch in enumerate(data_load):
    optimizer.zero_grad()
    x = batch[1].to(torch.float32)
    y = batch[2].to(torch.float32)
    output = model(x)
    is_out_nan = bool(torch.isnan(output).any())
    is_in_nan = bool(torch.isnan(x).any())
    print("model yields NaN:", is_out_nan, ",", "input has NaN:", is_in_nan)
    if is_out_nan:
        print(colored("Input has Nans", "red"))
        break
    loss = F.mse_loss(y, output)
    print(loss.item())
    loss.backward()
    optimizer.step()
    
    if i == 100:
        break

Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 0 ends: 71991
model yields NaN: False , input has NaN: False
0.01662561483681202
model yields NaN: False , input has NaN: False
0.029860835522413254
model yields NaN: False , input has NaN: False
0.017857719212770462
model yields NaN: False , input has NaN: False
0.021620701998472214
model yields NaN: False , input has NaN: False
0.02684231661260128
model yields NaN: False , input has NaN: False
0.021465139463543892
model yields NaN: False , input has NaN: False
0.020347464829683304
model yields NaN: False , input has NaN: False
0.03029012866318226
model yields NaN: False , input has NaN: False
0.020278573036193848
model yields NaN: False , input has NaN: False
0.01630883663892746
model yields NaN: False , input has NaN: False
0.01730329915881157
model yields NaN: False , input has NaN: False
0.01596924103796482


In [6]:
runCount = 0
device = torch.device("cuda") 
epochs = 2
train_steps = 2

resultsPath = "../model_ouputs/"
mString = 'base_test_en'
train_path = "../data/data_generated/train/train_ips_omni_df.csv"
val_path = "../data/data_generated/val/val_ips_omni_df.csv"
test_path = "../data/data_generated/test/test_ips_omni_df.csv"

train_set = DatasetHist(train_path)
val_set = DatasetHist(val_path)
test_set = DatasetHist(test_path)


lrs = [1.0e-3,1.0e-2] #learning rate
weight_decays = [1.0e-4,1.0e-3] #weight decay regularization
dropouts = [0.3,0.2,0.4]  #dropouts
batch_sizes = [32,16,64]      # batchsize
gammas = [0.95,0.90,0.85]#,0.80] #learning rate scheduler, reduces the learning rate as training gets cloes to the minima

run_count = 0

startT = time.time()
for lr in lrs:
    for weight_decay in weight_decays:
        for dropout in dropouts:
            for batch_size in batch_sizes:
                for gamma in gammas:
                    start_training(train_set, val_set, test_set, train_steps, epochs)
                    print(lr, weight_decay, dropout, batch_size, gamma)
                    run_count += 1
print(time.time() - startT)          


Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 0 ends: 71991
Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 71994 ends: 77399
Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 77456 ends: 81211
bestCorr is defined
0 epoch: 1 ; loss 2.0313868522644043
torch.float32
1 epoch: 1 ; loss 2.7876498699188232
torch.float32
2 epoch: 1 ; loss 1.9825559854507446
torch.float32
Run 0 [Epoch 1] loss: 4.0754e+01
val loss tensor(1.1112)
val loss tensor(2.2841)
val loss tensor(2.0220)
val loss tensor(1.2858)
val loss tensor(2.2990)
val loss tensor(1.2306)
val loss tensor(1.7049)
val loss tensor(1.5523)
val loss tensor(2.3337)


KeyboardInterrupt: 

In [5]:
def start_training(train_set, val_set, test_set, train_steps, epochs):
    # Get data loaded
    train_load = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_load = DataLoader(val_set, batch_size=batch_size, shuffle=True)
    test_load = DataLoader(test_set, batch_size=len(test_set), shuffle=False)

    # Load model
    model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=dropout, device=None)
    # Use GPU
    # model = model.to(device)

    # Inititialize optimizer ans scheduler
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

    train_Loss_y = np.empty(epochs,dtype=np.float32)
    train_Loss_diff = np.empty(epochs,dtype=np.float32)
    train_Loss = np.empty(epochs,dtype=np.float32)
    val_Loss_y = np.empty(epochs,dtype=np.float32)
    val_Loss_diff = np.empty(epochs,dtype=np.float32)
    val_Loss  = np.empty(epochs,dtype=np.float32)

    bestCorr = [-1.0,-1.0,-1.0,-1.0]
    print('bestCorr is defined')
    bestMse = [999.0,999.0,999.0,999.0]
    bestEpoch = epochs
    paraString = 'lr%0.2e_wd%0.2e_drp%0.2f_bS%d_E%d_sR%0.2f'%(lr, weight_decay, dropout, batch_size, epochs, gamma)

    def train(data_load, epoch, run_count=run_count, train_steps=train_steps):
        model.train()
        running_loss = 0.0
        running_loss_y = 0.0
        for i, data in enumerate(data_load):
            # zero the parameter gradients
            optimizer.zero_grad()
            loss, y_loss = doStep(data)
            print(i, "epoch:", epoch, ";", "loss", loss.item())
            del data

            print(loss.dtype)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_loss_y += y_loss.item()
            if i == train_steps:
                break

        # print statistics
        running_loss = running_loss/train_steps
        running_loss_y = running_loss_y/train_steps
        print('Run %d [Epoch %d] loss: %.4e' %(run_count, epoch, running_loss_y*800.0))
        return running_loss, running_loss_y

    def doStep(data):
        x = data[1].to(torch.float32)
        # print("x", x)
        y = data[2].to(torch.float32)
        y_out = model(x)
        # print("Before backward:", torch.isnan(y_out).any()) 

        loss = F.mse_loss(y_out, y, reduction='none')
        loss = loss * y * 100.0 # Why this scaling
        loss = loss.mean()
        y_loss = F.mse_loss(y_out,y)

        return loss, y_loss

    def validate(val_steps, epoch, run_count):
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            running_loss_y = 0.0
            for i, data in enumerate(val_load):
                loss, y_loss = doStep(data)
                print("val loss", loss)
                del data
    
                running_loss += loss.item()
                running_loss_y += y_loss.item()
                if i == val_steps:
                    break
    
            # print statistics
            running_loss = running_loss/val_steps
            running_loss_y = running_loss_y/val_steps
            print('Run %d [Epoch %d] val loss: %.4e' %(run_count, epoch, running_loss_y*800.0))
        return running_loss, running_loss_y

    def evalModel(bestCorr, bestMse, bestEpoch):
        model.eval()
        with torch.no_grad():
            for batch in test_load:
                testBatch = batch
                break
            x = testBatch[1].to(torch.float32)
            # y = testBatch[2]
            opY = model(x).detach().cpu().numpy()

        refY = testBatch[2].numpy()
        print("refY.shape", refY.shape)

        corrVals = [-1.0,-1.0,-1.0,-1.0]
        mseVals = [300.0,300.0,300.0,300.0]
        refIds = [9,11,13,15]
        for i in range(4):
            thisRef = refY[:, :, refIds[i]].flatten()
            thisOp = opY[:, :, refIds[i]].flatten()
            corrVals[i] = pearsonr(thisRef,thisOp)[0]
            mseVals[i] = np.mean(np.sqrt((thisOp-thisRef)**2)*800.0)
        if corrVals[-1] > bestCorr[-1]:
            bestCorr = corrVals
            torch.save(model.state_dict(), resultsPath + 'models/%s_%s'%(mString,paraString))
            bestMse = mseVals
            bestEpoch = epoch
        print(epoch, bestCorr, bestEpoch)
        return bestCorr, bestMse, bestEpoch  

    for epoch in range(1, epochs + 1):
        tlosses = train(train_load, epoch, run_count, train_steps)
        vlosses = validate(len(val_set)//batch_size, epoch, run_count)
        scheduler.step()

        train_Loss[epoch-1] = tlosses[0]
        val_Loss[epoch-1] = vlosses[0]
        
        train_Loss_y[epoch-1] = tlosses[1]
        val_Loss_y[epoch-1] = vlosses[1]
        
        if epoch > 1:
    	    bestCorr, bestMse, bestEpoch = evalModel(bestCorr, bestMse, bestEpoch)
    
    train_Loss.dump(resultsPath + 'losses/%s_train_%s'%(mString,paraString))
    val_Loss.dump(resultsPath + 'losses/%s_val_%s'%(mString,paraString))
    train_Loss_y.dump(resultsPath + 'losses/%s_train_y_%s'%(mString,paraString))
    val_Loss_y.dump(resultsPath + 'losses/%s_val_y_%s'%(mString,paraString))
    with open(resultsPath + 'corrsM2e','a') as fl:
        #fl.write('%s_%s\t%0.6f\t%0.6f\t%d\n'%(mString,paraString,bestCorr,bestMse,bestEpoch))
        fl.write('%s_%s'%(mString,paraString))
        for i in range(4):
            fl.write('\t%0.6f\t%0.6f'%(bestCorr[i],bestMse[i]))
        fl.write('\t%d\n'%bestEpoch)
    print(lr, time.time() - startT)
    return

In [61]:
train_df = pd.read_csv(train_path)

In [66]:
(train_df.loc[:, train_df.columns != "idx"].min() < 0).sum()

np.int64(0)